# 00 - Setup & Shared Helpers for the Advanced API

This folder demonstrates Infrared's **advanced** simulation inputs - the
additive fields the public `infrared-sdk` Pydantic models do **not** yet
expose, sent by hand-building the async JSON payload and POSTing it to the
*same* async endpoints the SDK uses.

**What "advanced" adds**

| Field | Models | What it does |
|---|---|---|
| `analysis-surfaces` (`facades`/`roofs`/`all`) | SR, DA, DSH, SVF | synthesize a UV sensor grid on building facades / roofs |
| `sensor-points` / `sensor-normals` | SR, DA, DSH, SVF | bring-your-own measurement locations |
| `ground-geometry` | all (incl. UTCI/TCS) | terrain mesh - **seats (conforms) BYO geometry onto the terrain**, occludes on sensor paths, drapes the grid |
| `terrain-alignment` (`auto-align`/`assume-aligned`) | all (with `ground-geometry`) | how the scene is seated: `auto-align` (default) seats misaligned objects onto the terrain; `assume-aligned` validates only → **422** if not pre-seated |
| `context-geometry` | SR, DA, DSH, SVF | occluder-only geometry (shadows in, no sensors) |
| `physics:"advanced"` | UTCI | hourly time-stepping thermal tier + canopy ray-march |
| multi-month / annual `time-period` | SR, DA, DSH, UTCI, TCS | windows beyond a single month |

**This notebook** just shows the shared helper modules every other notebook
imports. Read it once, then jump to a per-model notebook.

## Configuration (environment variables - never hard-coded)

* `INFRARED_API_KEY` - your key. Request access at <https://infrared.city>.
* `INFRARED_BASE_URL` - optional. Defaults to **staging**
  `https://api-test.infrared.city` (host root, **no** `/v2`), where the
  advanced features are deployed first.
  For **production** set `https://api.infrared.city/v2` (everything under `/v2`).

> The single base URL drives both the SDK client (geometry fetch) and the
> hand-built async POSTs. Get the `/v2` right for your environment (staging =
> no `/v2`, prod = `/v2`); the wrong one 404s everywhere.

In [1]:
# --- Setup: auth, base URL, geometry (self-contained) -----------------------
# Set your key in the environment first:  export INFRARED_API_KEY=...
# Optionally load a .env file (pip install python-dotenv):
try:
    from dotenv import load_dotenv

    load_dotenv()
except Exception:
    pass

import os

# Default base URL = STAGING (host root, NO /v2) where advanced features deploy
# first. For production set INFRARED_BASE_URL=https://api.infrared.city/v2
os.environ.setdefault("INFRARED_BASE_URL", "https://api-test.infrared.city")

import ir_advanced as ia
import ir_render as ir

print("base URL :", ia.base_url())
client = ia.make_client()
buildings = ia.fetch_buildings(
    client, ia.VIENNA_KARLSPLATZ, "karlsplatz_buildings.json"
)
print(f"buildings: {len(buildings)} (Vienna Karlsplatz AOI, fetched via SDK + cached)")

base URL : https://api-test.infrared.city


buildings: 132 (Vienna Karlsplatz AOI, fetched via SDK + cached)


## The two helper modules

* **`ir_advanced.py`** - client/auth, geometry fetch+cache (relative to this
  folder), the direct-API wire contract (`submit -> wait -> fetch_results`,
  combined as `run_job`), and `reconstruct_cells()` which turns an
  `analysis-surfaces` result into a world-space colored triangle soup (one
  clipped `cell-tris` footprint per kept cell).
* **`ir_render.py`** - `surface_mesh()` (Lambert-shaded colored cell mesh on the
  building geometry in 3D), `grid_heatmap()` (2D heatmap for the 512x512 grid
  models), `terrain_3d()`, and `footprints_2d()` (plan view).

Everything is self-contained: geometry is fetched through the public SDK and
cached under `./.cache/`, so these notebooks run cold for any SDK user.

In [2]:
# Quick tour of what's available.
print("ir_advanced:", [x for x in dir(ia) if not x.startswith("_") and x[0].islower()])
print()
print("ir_render  :", [x for x in dir(ir) if not x.startswith("_") and x[0].islower()])
print()
print("AOI polygon (Vienna Karlsplatz):")
print(ia.VIENNA_KARLSPLATZ["coordinates"][0])
w, h = ia.aoi_bounds_local(ia.VIENNA_KARLSPLATZ)
print(f"AOI local extent ~ {w:.0f} m (E-W) x {h:.0f} m (S-N)")

ir_advanced: ['annotations', 'aoi_bounds_local', 'api_key', 'async_base', 'base_url', 'building_faces', 'building_triangles', 'fetch_buildings', 'fetch_ground_materials', 'fetch_results', 'fetch_vegetation', 'fetch_weather_identifier', 'gzip', 'io', 'json', 'make_client', 'np', 'os', 'reconstruct_cells', 'requests', 'run_job', 'submit', 'time', 'wait', 'zipfile']

ir_render  : ['annotations', 'footprints_2d', 'grid_heatmap', 'matplotlib', 'np', 'plt', 'surface_mesh', 'terrain_3d']

AOI polygon (Vienna Karlsplatz):
[[16.368, 48.2], [16.371, 48.2], [16.371, 48.2018], [16.368, 48.2018], [16.368, 48.2]]
AOI local extent ~ 223 m (E-W) x 200 m (S-N)


## The async wire contract (what `run_job` does under the hood)

```
submit  : POST {base}/async/{type}    body = zip(payload.json)  -> 202 {jobId}
wait    : GET  {base}/async/jobs/{id}                            -> {jobStatus}
results : GET  {base}/async/jobs/{id}/results  -> presigned URL in Link header
download: GET  <presigned>             -> zip / gzip / raw JSON of the result
```

The advanced fields ride **alongside** the normal model request fields
(`geometries`, `vegetation`, `ground-materials`, weather arrays,
`time-period`, `latitude`, `longitude`). A payload with none of them is the
unchanged grid request. All advanced fields are **kebab-case** on the wire.

Next: open `01_solar_radiation_advanced.ipynb`.